# Procedural Graphs: Multi-Hop Reasoning on HotpotQA
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weilinear/paper_reading/blob/main/notebooks/02_hotpotqa_procedural_graph.ipynb)

Reproduction of HotpotQA multi-hop reasoning from **'Procedural Graphs: Self-Evolving Execution Structures for LLM Agents'** (Lu et al., 2026).
Compares:
1. **Vanilla ReAct** (Unguided baseline)
2. **Procedural Graph (Mode 1: Hand-crafted Expert Prior)**
3. **Procedural Graph (Mode 5: Scratch + Online Evolution)**


In [ ]:
# 1. Setup Environment
![ -d paper_reading ] || git clone https://github.com/weilinear/paper_reading.git
%cd paper_reading


In [ ]:
# 2. Inspect Procedural Graph Representation
from agents.procedural_graph import get_hotpotqa_expert_graph

expert_g = get_hotpotqa_expert_graph()
print(f"Expert Graph contains {len(expert_g.nodes)} nodes and {len(expert_g.edges)} edges.")

# Inspect localized 2-hop context around Step 1
print(expert_g.format_serialized_context("First_Hop_Retrieve", h=2))


In [ ]:
# 3. Run Benchmark Evaluation on Official HotpotQA Samples
from benchmarks.hotpotqa import HotpotQABenchmark, HotpotQAEnv
from evaluations.run_hotpotqa import run_evaluation, build_simulated_llm

bm = HotpotQABenchmark()
env = HotpotQAEnv()
llm = build_simulated_llm()

# Run evaluation across all 3 modes
results = run_evaluation(bm, env, llm, mode="all", split="dev", limit=20)


In [ ]:
# 4. Compare Sample Trajectories
# Observe how Vanilla ReAct cuts corners on Hop 1, whereas Procedural Graph enforces the 2nd hop
from agents.react import ReActAgent
from agents.procedural_graph import OnlineGuidanceEngine

task = bm.load_data(split="dev", limit=1)[0]
solver = ReActAgent(llm=llm)

print("--- [Task Question] ---")
print(task.question)
print("Gold Answer:", task.gold_answer)

# Run Vanilla
env.reset()
traj_v = solver.solve(task, env)
print("\n=== Vanilla ReAct Trajectory (Unguided) ===")
print(traj_v.format_trajectory())
print("Final Prediction:", traj_v.final_answer)

# Run with Procedural Graph
engine = OnlineGuidanceEngine(graph=expert_g, h_hops=2)
provider = lambda traj, act: engine.get_guidance(traj.question, traj.format_trajectory(), act, step_count=len(traj.steps))
env.reset()
traj_pg = solver.solve(task, env, guidance_provider=provider)
print("\n=== Procedural Graph Trajectory (Guided) ===")
print(traj_pg.format_trajectory())
print("Final Prediction:", traj_pg.final_answer)
